In [ ]:
# %%
import jax, jax.numpy as jnp
from tests.utils import Generator
from slimdqn.algorithms.dqn import DQN
from slimdqn.algorithms.dqnrcshared import DQNRCShared
from slimdqn.algorithms.idqnshared import iDQNShared
from slimdqn.algorithms.gidqnshared import GiDQNShared

KEY             = jax.random.PRNGKey(0)
N_ACTIONS       = 10
N_BELLMAN_ITERS = 5
LR, GAMMA       = 6.25e-5, 0.99

ARCHITECTURES = {
    "large": dict(features=[32, 64, 64, 512], observation_dim=(84, 84, 4), batch_size=32),  # Dopamine / Nature-CNN
    "small": dict(features=[16, 256],         observation_dim=(16, 16, 2), batch_size=16),  # downscaled
}
ALGO_ORDER = ["dqn", "qrc", "idqn", "gidqn"]

count_params = lambda t: sum(x.size   for x in jax.tree.leaves(t))
tree_bytes   = lambda t: sum(x.nbytes for x in jax.tree.leaves(t))


def build_algos(features, obs):
    return {
        "dqn":   (DQN(KEY, obs, N_ACTIONS, features, LR, GAMMA, 1, 1, 8000), True),
        "qrc":   (DQNRCShared(KEY, obs, N_ACTIONS, features, LR, GAMMA, 1, 0.25, 8000, 1), False),
        "idqn":  (iDQNShared(KEY, obs, N_ACTIONS, N_BELLMAN_ITERS, features, LR, GAMMA, 1, 0.25, 8000), True),
        "gidqn": (GiDQNShared(KEY, obs, N_ACTIONS, N_BELLMAN_ITERS, features, LR, GAMMA, 1, 0.25, 8000, 1), True),
    }

def measure(q, has_target, gen, batch_size):
    key = jax.random.PRNGKey(0)
    if has_target:
        learn = jax.jit(q.learn_on_batch).lower(
            q.params, q.target_params, q.optimizer_state, gen.samples(key), jnp.ones(batch_size)
        ).compile()
    else:
        learn = jax.jit(q.learn_on_batch).lower(
            q.params, q.optimizer_state, gen.samples(key), jnp.ones(batch_size)
        ).compile()

    return {
        "flops":        learn.cost_analysis()[0]["flops"],
        "num_params":   count_params(q.params) + (count_params(q.target_params) if has_target else 0),
        "memory_bytes": tree_bytes(q.params) + tree_bytes(q.optimizer_state)
                        + (tree_bytes(q.target_params) if has_target else 0),
    }


results = {}
for arch, cfg in ARCHITECTURES.items():
    gen = Generator(cfg["batch_size"], cfg["observation_dim"], N_ACTIONS)
    results[arch] = {
        name: measure(q, has_t, gen, cfg["batch_size"])
        for name, (q, has_t) in build_algos(cfg["features"], cfg["observation_dim"]).items()
    }

print(results)

In [ ]:
AVAILABLE_COLORS = {
    "black": "#000000",
    "blue": "#1F77B4",
    "light_blue": "#AEC7E8",
    "orange": "#FF7F0E",
    "light_orange": "#FFBB78",
    "green": "#2CA02C",
    "light_green": "#98DF8A",
    "red": "#D1797A",
    "light_red": "#FF9896",
    "purple": "#9467BD",
    "light_purple": "#C5B0D5",
    "brown": "#8C564B",
    "light_brown": "#C49C94",
    "pink": "#E377C2",
    "light_pink": "#F7B6D2",
    "grey": "#7F7F7F",
    "light_grey": "#C7C7C7",
    "yellow": "#DEDE00",
    "light_yellow": "#F0E886",
    "cyan": "#17BECF",
    "light_cyan": "#9EDAE5",
}


In [ ]:
# # %%
# import numpy as np
# import matplotlib.pyplot as plt
#
# SCALE       = 1.0
#
#
# ALGO_ORDER  = ["dqn", "qrc", "idqn", "gidqn"]
# ALGO_LABELS = {"dqn": "DQN", "qrc": "QRC", "idqn": "i-DQN", "gidqn": "Gi-DQN"}
#
# ARCH_ORDER  = ["large", "small"]
# ARCH_LABELS = {"large": "Atari  (84x84x4)", "small": "downscaled  (16x16x2)"}
# ARCH_COLORS = {"large": "#2f5c8a", "small": "#9ec3e6"}
#
# PANELS = [
#     ("flops",        "Training FLOPs",  "FLOPs per learning step"),
#     ("num_params",   "Parameter count", "number of parameters"),
#     ("memory_bytes", "Memory",          "network memory"),
# ]
#
# BAR_WIDTH     = 0.40    # width of each bar (group slot is 1.0)
# BAR_LABEL_REL = 0.78    # bar-top number size, relative to FONT_BASE
# BAR_LABEL_ROT = 0       # rotate bar-top numbers (e.g. 90 if crowded)
# Y_HEADROOM    = 1.20    # top y-limit = tallest bar * this (space for the numbers)
# EDGE_WIDTH    = 0.5     # bar outline width
#
#
# def fmt_si(v):
#     for u in ["", "K", "M", "G", "T"]:
#         if abs(v) < 1000:
#             return f"{v:.0f}{u}" if abs(v - round(v)) < 1e-9 else f"{v:.1f}{u}"
#         v /= 1000.0
#     return f"{v:.1f}P"
#
#
# def fmt_bytes(v):
#     for u in ["B", "KiB", "MiB", "GiB"]:
#         if v < 1024:
#             return f"{v:.1f} {u}"
#         v /= 1024.0
#     return f"{v:.1f} TiB"
#
#
# FORMATTERS = {"flops": fmt_si, "num_params": fmt_si, "memory_bytes": fmt_bytes}
#
# f = 11 * SCALE
# plt.rcParams.update({
#     "font.size": f, "axes.titlesize": f * 1.15, "axes.labelsize": f,
#     "xtick.labelsize": f * 0.95, "ytick.labelsize": f * 0.90,
#     "legend.fontsize": f * 0.95, "axes.axisbelow": True,
# })
#
# fig, axes = plt.subplots(1, len(PANELS), figsize=(15.0 * SCALE, 4.3 * SCALE))
# x = np.arange(len(ALGO_ORDER))
#
# for ax, (key, title, ylabel) in zip(axes, PANELS):
#     fmt, tallest = FORMATTERS[key], 0
#     for i, arch in enumerate(ARCH_ORDER):
#         offset = (i - (len(ARCH_ORDER) - 1) / 2) * BAR_WIDTH
#         vals = [results[arch][a][key] for a in ALGO_ORDER]
#         tallest = max(tallest, max(vals))
#         bars = ax.bar(x + offset, vals, BAR_WIDTH, label=ARCH_LABELS[arch],
#                       color=ARCH_COLORS[arch], edgecolor="black", linewidth=EDGE_WIDTH * SCALE)
#         for b, v in zip(bars, vals):
#             ax.text(b.get_x() + b.get_width() / 2, b.get_height(), fmt(v),
#                     ha="center", va="bottom", rotation=BAR_LABEL_ROT, fontsize=f * BAR_LABEL_REL)
#     ax.set_title(title, pad=8 * SCALE)
#     ax.set_ylabel(ylabel)
#     ax.set_xticks(x)
#     ax.set_xticklabels([ALGO_LABELS[a] for a in ALGO_ORDER])
#     ax.set_ylim(0, tallest * Y_HEADROOM)
#     ax.grid(axis="y", linewidth=0.5 * SCALE, alpha=0.35)
#     ax.spines[["top", "right"]].set_visible(False)
#
# handles, labels = axes[0].get_legend_handles_labels()
# fig.legend(handles, labels, loc="upper center", ncol=len(ARCH_ORDER),
#            frameon=False, bbox_to_anchor=(0.5, 1.005))
# fig.suptitle("Compute and memory footprint: Atari-scale vs. downscaled networks", y=1.10, fontsize=f * 1.4)
# fig.tight_layout(rect=[0, 0, 1, 0.94])
# plt.show()

In [ ]:
# %%
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from matplotlib.patches import Patch
from matplotlib.ticker import MaxNLocator

plt.rc("font", family="STIXGeneral", serif="Times New Roman", size=10)
plt.rc("mathtext", fontset="stix")
plt.rcParams.update({
    "axes.linewidth": 0.4, "lines.linewidth": 1.5,
    "xtick.major.width": 0, "ytick.major.width": 0.45,
    "xtick.major.size": 0, "ytick.major.size": 1.8, "grid.linewidth": 0.45,
})


ALGOS = {"dqn": "DQN", "qrc": "QRC", "idqn": "i-DQN", "gidqn": "Gi-DQN"}
ARCHS    = ["large", "small"]
METRICS  = [
    ("flops",        "FLOPs"),
    ("num_params",   "Number of Parameters"),
    ("memory_bytes", "Memory (MiB)"),
]
color = lambda a: {"dqn": "black", "qrc": "purple", "idqn": "orange", "gidqn": "blue"}[a]
light = lambda c: tuple(1 - (1 - x) * (1 - 0.35) for x in mcolors.to_rgb(c))

# formatter for rounding numbers on the bars
def fmt(v):                                   # NEW
    for u in ["", "K", "M", "G"]:
        if abs(v) < 1000:
            return f"{v:.0f}{u}"
        v /= 1000.0
    return f"{v:.0f}T"

fig = plt.figure(figsize=(5.5, 2.15))
gs = fig.add_gridspec(1, 3, wspace=0.18)
x = np.arange(len(ALGOS))

for i, (key, title) in enumerate(METRICS):
    ax = fig.add_subplot(gs[i])
    vmax = max(results[ar][al][key] for ar in ARCHS for al in ALGOS.keys()) if not key=="memory_bytes" else max(results[ar][al][key]/ 1024**2 for ar in ARCHS for al in ALGOS.keys())
    for k, arch in enumerate(ARCHS):
        for j, algo in enumerate(ALGOS):
            c = AVAILABLE_COLORS[color(algo)]
            res = results[arch][algo][key] if not key == "memory_bytes" else results[arch][algo][key] / 1024**2
            bar = ax.bar(x[j] + (k - 0.5) *  0.40, res, width= 0.40,
                   color=(c if arch == "large" else light(c)), zorder=3)[0]

            xc = bar.get_x() + bar.get_width() / 2
            if arch == "large":
                ax.text(xc, res * 0.97, fmt(res), ha="center", va="top",
                        rotation=90, fontsize=8.5, zorder=4,
                        color="white" if algo == "dqn" else "black")
            else:
                ax.text(xc, res + 0.02 * vmax, fmt(res), ha="center", va="bottom",
                        rotation=90, fontsize=8.5, zorder=4, color="black")
    ax.grid(axis="y", zorder=0)
    ax.set_xticks(x)
    ax.set_xticklabels([a for a in ALGOS.values()], rotation=45, ha="center", fontsize=8)
    ax.tick_params(axis="x", pad=1)
    ax.set_xlim(-0.6, len(ALGOS) - 0.4); ax.set_ylim(0, vmax * 1.3)
    ax.yaxis.set_major_locator(MaxNLocator(nbins=4))
    ax.set_title(title, fontsize=10, pad=6)
    if key != "memory_bytes":
        ax.ticklabel_format(axis="y", style="sci", scilimits=(0, 0), useMathText=True)
        #ax.yaxis.get_offset_text().set_fontsize(8)
        ax.yaxis.get_offset_text().set_visible(False)

algo_leg = [Patch(edgecolor="white", facecolor=AVAILABLE_COLORS[color(a)]) for a in ALGOS]
fig.legend(algo_leg, [a for a in ALGOS.values()], ncols=4, frameon=False, loc="center",
           bbox_to_anchor=(0.5, 0.94), columnspacing=1.5, handlelength=1,
           handletextpad=0.5, labelspacing=0.2, fontsize=12)

arch_leg = [Patch(facecolor="0.2"), Patch(facecolor=light("0.2"))]
fig.legend(arch_leg, ["Large Agent", "Scaled-Down Agent"], ncols=2, frameon=False, loc="center",
           bbox_to_anchor=(0.5, 0.06), columnspacing=1.5, handlelength=1,
           handletextpad=0.5, fontsize=9.5)

fig.subplots_adjust(left=0.06, right=0.985, bottom=0.34, top=0.73)
fig.savefig("footprint_grouped.pdf", pad_inches=0)

# place the x10^something somewhere else
fig.canvas.draw()
for ax in fig.axes:
    offset_text = ax.yaxis.get_offset_text()
    if offset_text.get_text():
        sci_text = offset_text.get_text()
        ax.text(0.0, 0.85, sci_text, transform=ax.transAxes,
                ha='right', va='bottom', fontsize=8)
fig.savefig("footprint_grouped.pdf", pad_inches=0, dpi=300)
fig